In [2]:
import pandas as pd

data_file = "/mnt/data_server/home/stu_zyb/MyQuant/download_data/csi300_2025-01-05_2025-01-08_M.parquet"

data = pd.read_parquet(data_file)

data

,date,code,open,high,low,close,volume,time
2025-01-06 09:35:00,2025-01-06,sh.600000,9.831844,9.831844,9.705670,9.744493,6003100,None
2025-01-06 09:40:00,2025-01-06,sh.600000,9.754198,9.763904,9.686259,9.734787,3484500,None
2025-01-06 09:45:00,2025-01-06,sh.600000,9.725081,9.744493,9.695964,9.734787,1382300,None
2025-01-06 09:50:00,2025-01-06,sh.600000,9.734787,9.734787,9.705670,9.705670,1396200,None
2025-01-06 09:55:00,2025-01-06,sh.600000,9.705670,9.715376,9.657142,9.686259,4219900,None
...,...,...,...,...,...,...,...,...
2025-01-08 14:40:00,2025-01-08,sz.301269,108.078275,108.387926,107.478950,108.387926,109900,None
2025-01-08 14:45:00,2025-01-08,sz.301269,108.387926,108.537757,108.278050,108.308016,72500,None
2025-01-08 14:50:00,2025-01-08,sz.301269,108.308016,108.677600,108.278050,108.377938,121500,None
2025-01-08 14:55:00,2025-01-08,sz.301269,108.377938,108.427881,108.098253,108.288039,128700,None


In [33]:
from concurrent.futures import ThreadPoolExecutor

def _long_to_wide_(long_data: pd.DataFrame, value_col: str) -> pd.Series:
    
    if 'code' not in long_data:
        raise ValueError('')
    
    wide_data = long_data.pivot(values = value_col, columns = 'code')
    
    return wide_data

def MPC_(minute_df):
    
    wide_close = _long_to_wide_(minute_df, 'close')

    MPC = wide_close / wide_close.shift(1) - 1
    
    MPC_max = MPC.max()
    
    MPC_max.index.name = 'asset_id'
    MPC_max.name = minute_df['date'].iloc[0]
    
    return MPC_max

def _calc_daily_factor_(data, date):
    
    if 'date' not in data or 'code' not in data:
        raise ValueError('')
    
    daily_df = data[data['date'] == date]
    
    if daily_df.empty:
        raise ValueError(f"No data for date {date}")
    
    MPC_max = MPC_(daily_df)
    
    return MPC_max

daily_factors = []

dates = data['date'].unique()
with ThreadPoolExecutor(max_workers=16) as pool:
    daily_factors = list(pool.map(lambda d: _calc_daily_factor_(data, d), dates))


In [32]:
pd.concat(daily_factors, axis=1)

,2025-01-06,2025-01-07,2025-01-08
asset_id,,,
sh.600000,0.010101,0.003953,0.004864
sh.600009,0.006073,0.002433,0.003070
sh.600010,0.011364,0.011050,0.011364
sh.600011,0.007680,0.003091,0.003091
sh.600015,0.007979,0.005256,0.003927
...,...,...,...
sz.300979,0.015310,0.010170,0.011920
sz.300999,0.006823,0.005193,0.004292
sz.301236,0.009675,0.006192,0.009128
